# Introduction to Probabilistic Programming Languages
## A guide in understanding higher order programs and the message interface
**By Julieta Cavalieri and Ema Sapirstein**



This guide is the final project of the course "Introduction to Probabilistic Programming" dictated by Javier Burroni in the Universidad de Buenos Aires during July 2026. It is based on chapters 5 and 6 of the paper ***van de Meent, J.-W., Paige, B., Yang, H., & Wood, F. (2018). An Introduction to Probabilistic Programming.*** arXiv:1809.10756.

The main purpose of this guide is to show an object-oriented implementation of the message interface that higher-order programs utilize, improving the June-26 lesson of the course.

Run cells top to bottom. At the end, test compares the results with the exact numbers.

To begin with, we clone the course's repository.

In [1]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-26

Cloning into 'IntroPPLs26'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 327 (delta 6), reused 13 (delta 4), pack-reused 308 (from 1)
Receiving objects: 100% (327/327), 43.82 MiB | 14.82 MiB/s, done.
Resolving deltas: 100% (163/163), done.
/content/IntroPPLs26/notebooks/Jun-26


We continue by importing the packages and functions needed.

In [2]:
import os, sys
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, 'interpreters', 'minippl')):
        sys.path.insert(0, os.path.join(_p, 'interpreters')); break
    _p = os.path.dirname(_p)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import softmax
from minippl import parse, Symbol, PRIMITIVES, is_primitive
from minippl.distributions import Normal, Bernoulli
from dataclasses import dataclass, replace, field

And we define the `log_prob` for the distributions utilized.
After this guide, you can try running the code with other distributions.

In [3]:
def log_prob(d, x):
    if isinstance(d, Normal):
        return float(stats.norm.logpdf(x, d.mu, d.sigma))
    if isinstance(d, Bernoulli):
        return float(stats.bernoulli.logpmf(int(x), d.p))
    raise NotImplementedError(f"no density for {d!r}")

## Dataclasses

We continue by defining the dataclasses needed to run this code. They represent the closure, the machine and all the First-class functions involved in a probabilistic language (evaluate, if, observe, sample, among others), in addition to the necessary ones to implement the message interface.

In [4]:
# First-class functions need closures: a value pairing a function body with the environment where it was defined
@dataclass(slots=True)
class Closure:
    params: list
    body: list
    env: dict

    def call(self, m, args, addr):
      new_env = dict(self.env)
      if len(self.params) != len(args):
        raise TypeError(f"arity mismatch: expected {len(self.params)}, got {len(args)}")
      for p, x in zip(self.params, args):
        new_env[p] = x
      _push_body(m.C, self.body, new_env, addr)


# Machine
@dataclass
class M:
  C: list
  env: dict
  rng: np.random.Generator
  V: list = field(default_factory=list)
  log_w: float = 0.0

  def resume(self):
      while self.C:
          instr = self.C.pop()
          result = instr.execute(self)
          if result is not None:
              return result
      return Done(self.V[-1], self)

  def send(self, value):
      self.V.append(value)

  def fork(self, rng=None):
      return M(C=self.C.copy(), V=self.V.copy(), env=self.env.copy(), rng=self.rng if rng is None else rng, log_w=self.log_w)

# Evaluate
@dataclass(slots=True)
class Ev:
    expr: object
    env: dict
    addr: tuple

    def execute(self, m):
      e = self.expr
      if isinstance(e, Symbol):
        if e in self.env:
          m.V.append(self.env[e])
        elif is_primitive(e):
          m.V.append(PRIMITIVES[e])
        else:
          raise NameError(e)
      elif hasattr(e, "eval"):
        e.eval(m, self.env, self.addr)
      else:
        m.V.append(e)


@dataclass(slots=True)
class IfK:
    then: object
    els: object
    env: dict
    addr: tuple

    def execute (self, m):
      branch, tag = (self.then, 'then') if m.V.pop() else (self.els, 'else')
      m.C.append(Ev(expr=branch, env=self.env, addr=self.addr + (tag,)))

@dataclass(slots=True)
class LetK:
    binds: list
    i: int
    body: list
    env: dict
    addr: tuple

    def execute(self, m):
      env = dict(self.env)
      env[self.binds[2*self.i]] = m.V.pop()
      if 2*(self.i+1) < len(self.binds):
        m.C.append(LetK(binds=self.binds, i=self.i+1, body=self.body, env=env, addr=self.addr))
        m.C.append(Ev(expr=self.binds[2*(self.i+1)+1], env=env, addr=self.addr + ('let', 2*(self.i+1))))
      else:
        _push_body(m.C, self.body, env, self.addr)

@dataclass(slots=True)
class CallK:
    nargs: int
    addr: tuple

    def execute(self, m):
      args = [m.V.pop() for _ in range(self.nargs)][::-1]
      f = m.V.pop()

      if hasattr(f, "call"):
        f.call(m, args, self.addr)
      else:
        m.V.append(f(*args))

@dataclass(slots=True)
class SampleK:
    addr: tuple

    def execute(self, m):
      d = m.V.pop()
      return SampleMsg(self.addr, d, m)

@dataclass(slots=True)
class ObserveK:
    addr: tuple

    def execute(self, m):
      y = m.V.pop()
      d = m.V.pop()
      return ObserveMsg(self.addr, d, y, m)

@dataclass(slots=True)
class Discard:

    def execute(self, m):
      m.V.pop()

@dataclass(slots=True)
class Let:
    binds: list
    body: list

    def eval(self, m, env, addr):
      if self.binds:
        m.C.append(LetK(self.binds, 0, self.body, env, addr))
        m.C.append(Ev(self.binds[1], env, addr + ('let', 0)))
      else:
        _push_body(m.C, self.body, env, addr)

@dataclass(slots=True)
class If:
    test: object
    then: object
    els: object

    def eval(self, m, env, addr):
      m.C.append(IfK(self.then, self.els, env, addr))
      m.C.append(Ev(self.test, env, addr + ('test',)))

@dataclass(slots=True)
class Fn:
    params: list
    body: list

    def eval(self, m, env, addr):
      m.V.append(Closure(self.params, self.body, dict(env)))

@dataclass(slots=True)
class Sample:
    dist: object

    def eval(self, m, env, addr):
      m.C.append(SampleK(addr))
      m.C.append(Ev(self.dist, env, addr + ('d',)))

@dataclass(slots=True)
class Observe:
    dist: object
    value: object

    def eval(self, m, env, addr):
      m.C.append(ObserveK(addr))
      m.C.append(Ev(self.value, env, addr + ('v',)))
      m.C.append(Ev(self.dist, env, addr + ('d',)))

@dataclass(slots=True)
class Call:
    fn: object
    args: list

    def eval(self, m, env, addr):
      m.C.append(CallK(len(self.args), addr))
      for i in range(len(self.args)-1, -1, -1):
        m.C.append(Ev(self.args[i], env, addr + (i - 1,)))
      m.C.append(Ev(self.fn, env, addr + ('fn',)))

@dataclass(slots=True)
class SampleMsg:
    addr: tuple
    dist: object
    machine: M

@dataclass(slots=True)
class ObserveMsg:
    addr: tuple
    dist: object
    value: object
    machine: M

@dataclass(slots=True)
class Done:
    value: object
    machine: M

@dataclass(slots=True)
class Msg:
    value: object
    machine: M

**The stack machine**

The control stack `C` and value stack `V` together are the continuation: pausing is returning from `resume()`, resuming is calling it again, forking is copying the stacks. `Closure()` is a first-class function value; `M()` is a suspendable execution.

**The parser**

The utilized parser returns a list `['let' , [..]]` and we need it as dataclasses nodes of an Abstract Syntax Tree (AST). The following function makes that transformation, to let the parser interact with the `resume()` function.

In [5]:
def parsed_to_ast(parsed):

    #variables as mu, +, etc.
    if isinstance(parsed, Symbol):
        return parsed

    # Number or boolean
    if not isinstance(parsed, list):
        return parsed  # literal: int, float, bool

    #We use the first element to find which expression are we facing
    head = parsed[0]
    if head == 'let':
        binds, *body = parsed[1:]
        new_binds = []
        for i in range(0, len(binds), 2):
            new_binds.append(binds[i])
            new_binds.append(parsed_to_ast(binds[i + 1]))
        return Let(binds=new_binds, body=[parsed_to_ast(e) for e in body])
    if head == 'if':
        _, test, then, els = parsed
        return If(test=parsed_to_ast(test), then=parsed_to_ast(then), els=parsed_to_ast(els))
    if head == 'sample':
        return Sample(dist=parsed_to_ast(parsed[1]))
    if head == 'observe':
        return Observe(dist=parsed_to_ast(parsed[1]), value=parsed_to_ast(parsed[2]))
    if head == 'fn':
        return Fn(params=parsed[1], body=[parsed_to_ast(e) for e in parsed[2:]])
    return Call(fn=parsed[0], args=[parsed_to_ast(arg) for arg in parsed[1:]])

## The message-interface runtime

`resume(m)` runs until the next probabilistic effect and returns one of:

```
SampleMsg(address, distribution, m)
ObserveMsg(address, distribution, observed_value, m)
Done(value, m)
```

**One runtime, many algorithms**

At `samplek` and `observek` the evaluator emits `sample` and `observe` messages and does no inference: it records the address and returns the message. The controller answers with `send(m, value)`, which pushes a value, and resumes.

## Improvement 1: method-based machine interface

As a design improvement, we refactored the original long `resume(m)` function into a more object-oriented structure. Instead of handling all expression and continuation cases inside one large function, each dataclass now implements the behavior that belongs to it: expressions define how they are evaluated, and continuations define how they continue the computation.

After this refactor, the resume loop only performs the generic machine step: it pops the next instruction, executes it, and returns a message when the program reaches a probabilistic effect or finishes. This made it natural to move `resume` and `send` into the `M` machine class.

This does not change the semantics of the interpreter or the inference algorithms. It only makes the runtime organization more object-oriented, while keeping resume(m) and send(m, value) as compatibility wrappers.

In [6]:
def _push_body(C, body, env, addr):
    seq = []
    for n, b in enumerate(body[:-1]):
        seq.append(Ev( b, env, addr + ('body', n)))
        seq.append(Discard())
    seq.append(Ev(body[-1], env, addr + ('body', len(body) - 1)))
    for item in reversed(seq):
        C.append(item)

def resume(m):
    return m.resume()

def send(m, value):
    m.send(value)

def initial_machine(program, rng):
    genv = {}
    main = None
    for form in parse(program):
        if isinstance(form, list) and form and form[0] == 'defn':
            _, name, params, *body = form
            genv[name] = Closure(params, [parsed_to_ast(b) for b in body], genv)
        else:
            main = parsed_to_ast(form)
    return M(C=[Ev(main, genv, ())], env=genv, rng=rng)

## Likelihood weighting and SMC controllers


Two controllers over the message stream. At `sample` LW draws from the prior; at `observe` it accumulates the log-weight. SMC keeps many machines, advancing each to its next breakpoint, then scoring, resampling, and forking.

## Improvement 2: strict validation in SMC

As a second improvement, we added a stricter validation step to `run_smc`.


In the original version, SMC only checked whether all particles had reached an `ObserveMsg`. However, this is not enough: different particles could reach different `observe` sites, especially in programs with stochastic control flow. In that case, resampling them together would be incorrect, because SMC assumes that all particles are synchronized at the same observation point.

To make this explicit, we added `validate_smc_messages(messages)`, which checks that all particles reached the same kind of breakpoint and, when they reach an `ObserveMsg`, that all of them reached the same observe address.


With this change, `run_smc` now fails with a clear error if particles are not synchronized. This makes the implementation safer and also documents one of the assumptions behind the SMC controller.

In [7]:
def run_lw(program, rng):
    m = initial_machine(program, rng)
    while True:
        msg = m.resume()
        if isinstance(msg, Done):
            return msg.value, msg.machine.log_w
        if isinstance(msg, SampleMsg):
            msg.machine.send(msg.dist.sample(msg.machine.rng))
        elif isinstance(msg, ObserveMsg):
            msg.machine.log_w += log_prob(msg.dist, msg.value)
            msg.machine.send(msg.value)

def likelihood_weighting(program, rng, N):
    values, log_w = zip(*(run_lw(program, rng) for _ in range(N)))
    return np.array(values, dtype=float), softmax(np.array(log_w))

def advance(m):
    msg = m.resume()
    while isinstance(msg, SampleMsg):
        msg.machine.send(msg.dist.sample(msg.machine.rng))
        msg = m.resume()
    return msg


def validate_smc_messages(messages):

    # SMC assumes that all particles stop at the same kind of breakpoint. In particular, when particles stop at an observe, they should all stop at the same observe address
    # This catches programs where different particles follow different control flow paths and reach different observe sites

    if all(isinstance(msg, Done) for msg in messages):
        return "done"

    if not all(isinstance(msg, ObserveMsg) for msg in messages):
        types = [type(msg).__name__ for msg in messages]
        raise ValueError(
            "SMC needs all particles to reach the same kind of breakpoint. "
            f"Got message types: {list(set(types))}"
        )

    addresses = [msg.addr for msg in messages]
    first = addresses[0]

    if not all(addr == first for addr in addresses):
        raise ValueError(
            "SMC needs all particles to reach the same observe address. "
            f"Got observe addresses: {list(set(addresses))}"
        )

    return "observe"


def run_smc(program, rngs, N):
    particles = [initial_machine(program, rngs[i]) for i in range(N)]

    while True:
        messages = [advance(p) for p in particles]

        status = validate_smc_messages(messages)

        if status == "done":
            return np.array([float(msg.value) for msg in messages])

        # If we get here, validation already checked that all messages are ObserveMsg and that they have the same observe address
        log_inc = []
        paused = []

        for msg in messages:
            lp = log_prob(msg.dist, msg.value)
            msg.machine.log_w += lp

            log_inc.append(lp)
            msg.machine.send(msg.value)
            paused.append(msg.machine)

        anc = rngs[0].choice(N, size=N, p=softmax(np.array(log_inc)))
        particles = [paused[i].fork(rng=rngs[j]) for j, i in enumerate(anc)]

---
## Single-site MH controller

Single-site MH keeps an **address-keyed trace**. Each step picks one address, **resamples** that site, **reuses** the rest by address, re-runs, and accepts on the Metropolis-Hastings ratio.

The dimension-corrected ratio `mh_log_alpha` is given (it is the single-site ratio with prior proposals: the proposal densities at the resampled and newly reached sites cancel against the prior, leaving the reused sites, the observes, and a `1/n` term for the change in trace length). The point of this activity is to see that the controller has enough information to compute the ratio from the two address-keyed traces.

In [8]:
def mh_log_alpha(X, X2, S, S2, O, O2, a0):
    """Single-site MH log acceptance ratio (given).
    X, X2  : address -> value, current and proposed traces
    S, S2  : address -> log_prob at sample sites
    O, O2  : address -> log_prob at observe sites
    a0     : the resampled address
    """
    fwd = {a0} | (set(X2) - set(X))     # resampled + newly required sites
    rev = {a0} | (set(X)  - set(X2))    # resampled + dropped sites
    num = sum(p for k, p in S2.items() if k not in fwd) + sum(O2.values())
    den = sum(p for k, p in S.items()  if k not in rev) + sum(O.values())
    return (np.log(len(X)) - np.log(len(X2))) + (num - den)

def run(program, rng, x0, cache):
    """One execution over the message interface.
    x0       : the address to redraw (or None)
    cache    : {address: value} from the current trace, to reuse
    Returns (value, X, S, O): trace values, sample log-probs, observe log-probs.
    """
    m = initial_machine(program, rng); X, S, O = {}, {}, {}
    while True:
        msg = m.resume()
        if isinstance(msg, SampleMsg):
            if msg.addr in cache and msg.addr != x0:
              x = cache[msg.addr]
            else:
              x = msg.dist.sample(rng)
            X[msg.addr] = x; S[msg.addr] = log_prob(msg.dist, x); msg.machine.send(x)
        elif isinstance(msg, ObserveMsg):
            O[msg.addr] = log_prob(msg.dist, msg.value); msg.machine.send(msg.value)
        elif isinstance(msg, Done):
            return msg.value, X, S, O

def single_site_mh(program, rng, steps, warmup=2000):
    value, X, S, O = run(program, rng, None, {})    # initial trace: nothing to resample or reuse
    chain = []
    for i in range(steps + warmup):
        a0 = list(X)[int(rng.integers(len(X)))]                       # pick one site to change
        v2, X2, S2, O2 = run(program, rng, a0, X)
        if np.log(rng.random()) < mh_log_alpha(X, X2, S, S2, O, O2, a0):
          value, X, S, O = v2, X2, S2, O2
        if i >= warmup:
            chain.append(float(value))
    return np.array(chain, dtype=float)

##Tests

### Test: conjugate and a discrete-sum model

Conjugate: prior $\mu \sim \mathcal{N}(0,1)$, likelihood $y\mid\mu \sim \mathcal{N}(\mu,1)$, $y=2.3$. Exact posterior $\mathcal{N}(1.15, 0.5)$.

Bit-sum: eight fair coins, `(observe (normal 7 2) total)`. Exact mean below.

In [9]:
conj = "(let [mu (sample (normal 0 1))] (observe (normal mu 1) 2.3) mu)"
ch = single_site_mh(conj, np.random.default_rng(0), 60000, warmup=3000)
print(f"conj   SSMH mean = {ch.mean():.3f}  std = {ch.std():.3f}   (exact 1.150, {0.5**0.5:.3f})")

import math
from math import comb
bits = "(let [" + " ".join(f"b{i} (if (sample (bernoulli 0.5)) 1 0)" for i in range(1, 9)) \
     + " total (+ " + " ".join(f"b{i}" for i in range(1, 9)) + ")]" \
     + " (observe (normal 7 2) total) total)"
w = {k: comb(8, k) * math.exp(-0.5 * ((k - 7) / 2) ** 2) for k in range(9)}
exact = sum(k * w[k] for k in w) / sum(w.values())
ch2 = single_site_mh(bits, np.random.default_rng(1), 40000, warmup=3000)
print(f"bits   SSMH mean = {ch2.mean():.3f}   (exact {exact:.3f})")

conj   SSMH mean = 1.144  std = 0.709   (exact 1.150, 0.707)
bits   SSMH mean = 4.987   (exact 5.014)


### One model, three controllers

The same conjugate posterior, recovered by LW, SMC, and SSMH over the one runtime.

In [10]:
vals, w_lw = likelihood_weighting(conj, np.random.default_rng(2), 100000)
lw_mean = float((w_lw * vals).sum())
smc = run_smc(conj, [np.random.default_rng(1000 + i) for i in range(20000)], 20000)
print(f"LW   mean = {lw_mean:.3f}")
print(f"SMC  mean = {smc.mean():.3f}")
print(f"SSMH mean = {ch.mean():.3f}    (all exact 1.150, one runtime)")

LW   mean = 1.148
SMC  mean = 1.131
SSMH mean = 1.144    (all exact 1.150, one runtime)


---
## Closure application

`geom` and the closure example need **user-defined functions**, which the runtime applies in the `callk` case. That case starts a new environment from the closure's captured `env` and binds the parameters. The bindings extend the **closure's** environment, not the caller's: that is lexical scope.


### Check: closures and recursion

A returned closure remembers `mu` (lexical scope), and the geometric program recurses a random number of times. These should print `13` and a mean near `2.33`.


In [11]:
shift = "(let [make-shift (fn [mu] (fn [x] (+ x mu)))  f (make-shift 10)] (f 3))"
print("closure: (f 3) =", run_lw(shift, np.random.default_rng(0))[0], " (expect 13)")

geom = "(defn geom [] (if (sample (bernoulli 0.3)) 0 (+ 1 (geom)))) (geom)"
rng = np.random.default_rng(1)
ks = np.array([run_lw(geom, rng)[0] for _ in range(200000)])
print(f"geom mean = {ks.mean():.3f}   exact (1-p)/p = {0.7/0.3:.3f}")

closure: (f 3) = 13  (expect 13)
geom mean = 2.331   exact (1-p)/p = 2.333
